# Afya MedQoL — exemplo de uso

Notebook interno com um exemplo de cálculo do índice Afya MedQoL usando a
biblioteca `afya_medqol` (calibração 2024_2).

Cobre:
1. Escoragem de um único respondente (dict de respostas).
2. Escoragem em lote a partir de um CSV.
3. Leitura das colunas de saída (theta, T-score).

## 1. Instalação e import

Se o pacote ainda não estiver instalado no ambiente:

In [1]:
# %pip install afya-medqol

In [2]:
import sys

sys.path.append("/Users/Marcela.Motta/Desktop/Projetos/afya_medqol/src")


In [3]:
import pandas as pd

from afya_medqol import MedQoLPhysicianCalculator

calc = MedQoLPhysicianCalculator()
print(calc.factor_items)

{'Factor1': ['F1_1_enjoymentoflife', 'F1_2_financialsufficiency', 'F1_3_accesstoinformation', 'F1_4_leisureopportunities', 'F1_5_mobilitypast2weeks', 'F1_6_accesstohealthservices'], 'Factor2': ['F2_1_technicaltraining', 'F2_2_mentalhealthsupport', 'F2_3_coworkersupportnetwork', 'F2_4_educationalhandlingoferrors'], 'Factor3': ['F3_1_stresshurtsperformance', 'F3_2_stressledtoerrors', 'F3_3_stresshurtsrelationships']}


In [4]:
perguntas_en = calc.item_questions(lang="pt")  
perguntas_en 

{'F1_1_enjoymentoflife': '28.3. O quanto você aproveita a vida?',
 'F1_2_financialsufficiency': '29.3. Você tem dinheiro suficiente para satisfazer suas necessidades?',
 'F1_3_accesstoinformation': '29.4. Quão disponíveis para você estão as informações que precisa no seu dia-a-dia?',
 'F1_4_leisureopportunities': '29.5. Em que medida você tem oportunidades de atividade de lazer?',
 'F1_5_mobilitypast2weeks': '30. Quão bem você é capaz de se locomover nas últimas duas semanas?',
 'F1_6_accesstohealthservices': '31.9. Quão satisfeito(a) você está com o seu acesso aos serviços de saúde?',
 'F2_1_technicaltraining': '59.2. A instituição em que trabalho, oferece treinamento técnico a equipe.',
 'F2_2_mentalhealthsupport': '59.3. A instituição em que trabalho, oferece suporte em caso de psicoadoecimento adoecimento mental ou sofrimento emocional.',
 'F2_3_coworkersupportnetwork': '59.4. Na instituição em que trabalho, sinto que posso contar com uma rede de apoio de meus colegas de trabalho.'

In [5]:
alternativas_pt = calc.item_options(lang="pt")
alternativas_pt

{'F1_1_enjoymentoflife': ['Nada (1)',
  'Muito pouco (2)',
  'Mais ou menos (3)',
  'Bastante (4)',
  'Extremamente (5)'],
 'F1_2_financialsufficiency': ['Nada (1)',
  'Muito pouco (2)',
  'Médio (3)',
  'Muito (4)',
  'Completamente (5)'],
 'F1_3_accesstoinformation': ['Nada (1)',
  'Muito pouco (2)',
  'Médio (3)',
  'Muito (4)',
  'Completamente (5)'],
 'F1_4_leisureopportunities': ['Nada (1)',
  'Muito pouco (2)',
  'Médio (3)',
  'Muito (4)',
  'Completamente (5)'],
 'F1_5_mobilitypast2weeks': ['Muito ruim (1)',
  'Ruim (2)',
  'Nem ruim, nem bom (3)',
  'Bom (4)',
  'Muito bom (5)'],
 'F1_6_accesstohealthservices': ['Muito insatisfeito (1)',
  'Insatisfeito (2)',
  'Nem satisfeito, nem insatisfeito (3)',
  'Satisfeito (4)',
  'Muito satisfeito (5)'],
 'F2_1_technicaltraining': ['Discordo totalmente (1)',
  'Discordo em parte (2)',
  'Não concordo nem discordo (3)',
  'Concordo em parte (4)',
  'Concordo totalmente (5)'],
 'F2_2_mentalhealthsupport': ['Discordo totalmente (1)',
  

## 2. Escoragem de um único respondente

`MedQoLPhysicianCalculator` precomputa a grade de quadratura e as probabilidades por
item uma única vez na construção — reutilize a mesma instância para escorar
vários respondentes/lotes.

In [6]:
calc = MedQoLPhysicianCalculator()

respostas_medico = {
    "F1_1_enjoymentoflife": 1,
    "F1_2_financialsufficiency": 4,
    "F1_3_accesstoinformation": 3,
    "F1_4_leisureopportunities": 4,
    "F1_5_mobilitypast2weeks": 3,
    "F1_6_accesstohealthservices": 4,
    "F2_1_technicaltraining": 3,
    "F2_2_mentalhealthsupport": 3,
    "F2_3_coworkersupportnetwork": 2,
    "F2_4_educationalhandlingoferrors": 3,
    "F3_1_stresshurtsperformance": 4,
    "F3_2_stressledtoerrors": 3,
    "F3_3_stresshurtsrelationships": 4,
}

resultado = calc.score_physician(respostas_medico)

for chave in (
    "theta1_quality_of_life", "theta2_institutional_support", "theta3_perceived_stress", "theta_global", "T_score_global",
):
    print(f"{chave:28s} {resultado[chave]}")

theta1_quality_of_life       -0.285195849224565
theta2_institutional_support 0.16229324407530046
theta3_perceived_stress      0.11841287222597136
theta_global                 -0.09064374474810055
T_score_global               49.093562552518996


## 3. Escoragem em lote (vários respondentes)

Um `DataFrame` com uma coluna por item (mesmos nomes de `ITENS_TODOS`) e uma
linha por respondente. Valores ausentes ou `999` (código de "não respondeu")
são tratados como omissos.

In [7]:
df_respostas = pd.DataFrame([
    {
        "physician_id": "M001",
        "F1_1_enjoymentoflife": 5, "F1_2_financialsufficiency": 5, "F1_3_accesstoinformation": 5,
        "F1_4_leisureopportunities": 5, "F1_5_mobilitypast2weeks": 5, "F1_6_accesstohealthservices": 5,
        "F2_1_technicaltraining": 5, "F2_2_mentalhealthsupport": 5,
        "F2_3_coworkersupportnetwork": 5, "F2_4_educationalhandlingoferrors": 5,
        "F3_1_stresshurtsperformance": 1, "F3_2_stressledtoerrors": 1, "F3_3_stresshurtsrelationships": 1,
    },
    {
        "physician_id": "M002",
        "F1_1_enjoymentoflife": 1, "F1_2_financialsufficiency": 1, "F1_3_accesstoinformation": 1,
        "F1_4_leisureopportunities": 1, "F1_5_mobilitypast2weeks": 1, "F1_6_accesstohealthservices": 1,
        "F2_1_technicaltraining": 1, "F2_2_mentalhealthsupport": 1,
        "F2_3_coworkersupportnetwork": 1, "F2_4_educationalhandlingoferrors": 1,
        "F3_1_stresshurtsperformance": 5, "F3_2_stressledtoerrors": 5, "F3_3_stresshurtsrelationships": 5,
    },
    {
        "physician_id": "M003",
        "F1_1_enjoymentoflife": 3, "F1_2_financialsufficiency": 3, "F1_3_accesstoinformation": 3,
        "F1_4_leisureopportunities": 3, "F1_5_mobilitypast2weeks": 3, "F1_6_accesstohealthservices": 3,
        "F2_1_technicaltraining": 3, "F2_2_mentalhealthsupport": 3,
        "F2_3_coworkersupportnetwork": 3, "F2_4_educationalhandlingoferrors": 3,
        "F3_1_stresshurtsperformance": 3, "F3_2_stressledtoerrors": 3, "F3_3_stresshurtsrelationships": 3,
    },
])

df_resultado = calc.score_batch(df_respostas)
df_resultado[["physician_id", "theta1_quality_of_life", "theta2_institutional_support", "theta3_perceived_stress",
              "theta_global", "T_score_global"]]

,physician_id,theta1_quality_of_life,theta2_institutional_support,theta3_perceived_stress,theta_global,T_score_global
0,M001,2.541674,2.173227,-1.882549,2.240849,72.408492
1,M002,-3.436587,-1.520615,1.639552,-2.311715,26.882845
2,M003,-0.434366,0.275357,-0.418584,0.032791,50.327912


In [8]:
import numpy as np
import pandas as pd

np.random.seed(42)

colunas = [
    "F1_1_enjoymentoflife",
    "F1_2_financialsufficiency",
    "F1_3_accesstoinformation",
    "F1_4_leisureopportunities",
    "F1_5_mobilitypast2weeks",
    "F1_6_accesstohealthservices",
    "F2_1_technicaltraining",
    "F2_2_mentalhealthsupport",
    "F2_3_coworkersupportnetwork",
    "F2_4_educationalhandlingoferrors",
    "F3_1_stresshurtsperformance",
    "F3_2_stressledtoerrors",
    "F3_3_stresshurtsrelationships",
]

def gerar_dataset(n_amostras=5000, random_state=42):

    rng = np.random.default_rng(random_state)

    dados = {}

    # Cada coluna recebe aproximadamente a mesma quantidade de notas 1-5
    for coluna in colunas:

        valores = np.tile(np.arange(1, 6), int(np.ceil(n_amostras/5)))[:n_amostras]
        rng.shuffle(valores)
        dados[coluna] = valores

    df = pd.DataFrame(dados)

    # Adiciona alguns perfis extremos
    extremos = pd.DataFrame([
        [1]*13,
        [2]*13,
        [3]*13,
        [4]*13,
        [5]*13,
        [5]*10 + [1]*3,
        [1]*10 + [5]*3,
        [5,1]*6 + [5],
        [1,5]*6 + [1],
    ], columns=colunas)

    df = pd.concat([extremos, df], ignore_index=True)

    df.insert(
        0,
        "physician_id",
        [f"M{i+1:05d}" for i in range(len(df))]
    )

    # Soma das respostas
    df["total_score"] = df[colunas].sum(axis=1)

    # Vetor/lista com as respostas
    df["responses_vector"] = df[colunas].values.tolist()

    return df


df_respostas = gerar_dataset(5000)

df_respostas

,physician_id,F1_1_enjoymentoflife,F1_2_financialsufficiency,F1_3_accesstoinformation,F1_4_leisureopportunities,F1_5_mobilitypast2weeks,F1_6_accesstohealthservices,F2_1_technicaltraining,F2_2_mentalhealthsupport,F2_3_coworkersupportnetwork,F2_4_educationalhandlingoferrors,F3_1_stresshurtsperformance,F3_2_stressledtoerrors,F3_3_stresshurtsrelationships,total_score,responses_vector
0,M00001,1,1,1,1,1,1,1,1,1,1,1,1,1,13,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
1,M00002,2,2,2,2,2,2,2,2,2,2,2,2,2,26,"[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]"
2,M00003,3,3,3,3,3,3,3,3,3,3,3,3,3,39,"[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]"
3,M00004,4,4,4,4,4,4,4,4,4,4,4,4,4,52,"[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]"
4,M00005,5,5,5,5,5,5,5,5,5,5,5,5,5,65,"[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5004,M05005,3,1,5,1,5,1,2,2,3,1,1,5,1,31,"[3, 1, 5, 1, 5, 1, 2, 2, 3, 1, 1, 5, 1]"
5005,M05006,2,5,1,4,1,5,4,3,3,3,5,4,4,44,"[2, 5, 1, 4, 1, 5, 4, 3, 3, 3, 5, 4, 4]"
5006,M05007,5,5,2,5,1,5,5,2,2,5,4,3,3,47,"[5, 5, 2, 5, 1, 5, 5, 2, 2, 5, 4, 3, 3]"
5007,M05008,3,2,4,1,3,5,4,5,1,2,1,1,1,33,"[3, 2, 4, 1, 3, 5, 4, 5, 1, 2, 1, 1, 1]"


In [9]:
df_resultado = calc.score_batch(df_respostas)
df_resultado[["physician_id", "total_score", "theta1_quality_of_life", "theta2_institutional_support", "theta3_perceived_stress",
              "theta_global", "T_score_global"]]

,physician_id,total_score,theta1_quality_of_life,theta2_institutional_support,theta3_perceived_stress,theta_global,T_score_global
0,M00001,13,-3.436587,-1.520615,-1.882549,-1.362429,36.375712
1,M00002,26,-1.728355,-0.222866,-0.859886,-0.527464,44.725358
2,M00003,39,-0.434366,0.275357,-0.418584,0.032791,50.327912
3,M00004,52,0.817756,0.948237,0.236383,0.577263,55.772627
4,M00005,65,2.541674,2.173227,1.639552,1.291563,62.915626
...,...,...,...,...,...,...,...
5004,M05005,31,-1.177422,-0.255440,-1.367332,-0.183325,48.166752
5005,M05006,44,0.105652,0.397066,0.687943,-0.010817,49.891826
5006,M05007,47,1.850626,0.722002,-0.137349,1.011548,60.115483
5007,M05008,33,-1.033885,0.164514,-1.882549,0.152798,51.527976


In [10]:
import plotly.express as px

fig = px.scatter(
    df_resultado,
    x="total_score",
    y="T_score_global",
    hover_name="responses_vector",  # aparece em destaque ao passar o mouse
    hover_data={
        "total_score": True,
        "T_score_global": True,
        "responses_vector": False,   # evita duplicação, pois já está em hover_name
    },
    title="Enjoyment of Life vs Total Score"
)

fig.update_layout(
    xaxis_title="Total Score",
    yaxis_title="T_score_global",
    template="plotly_white"
)

fig.show()

## 4. Lendo direto de um CSV com score_batch

`score_batch` só aceita um `DataFrame` já carregado — para pontuar direto
de um arquivo CSV, basta ler com `pandas.read_csv(...)` e passar o
resultado. Por ser uma biblioteca, ela não salva nada em disco — quem
quiser persistir o resultado usa `out.to_csv(...)` diretamente, ou a linha
de comando (`afya-medqol respostas.csv --saida resultado.csv`).

In [11]:
out = calc.score_batch(df_respostas)
out.head()

,physician_id,F1_1_enjoymentoflife,F1_2_financialsufficiency,F1_3_accesstoinformation,F1_4_leisureopportunities,F1_5_mobilitypast2weeks,F1_6_accesstohealthservices,F2_1_technicaltraining,F2_2_mentalhealthsupport,F2_3_coworkersupportnetwork,...,F3_1_stresshurtsperformance,F3_2_stressledtoerrors,F3_3_stresshurtsrelationships,total_score,responses_vector,theta1_quality_of_life,theta2_institutional_support,theta3_perceived_stress,theta_global,T_score_global
0,M00001,1,1,1,1,1,1,1,1,1,...,1,1,1,13,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]",-3.436587,-1.520615,-1.882549,-1.362429,36.375712
1,M00002,2,2,2,2,2,2,2,2,2,...,2,2,2,26,"[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]",-1.728355,-0.222866,-0.859886,-0.527464,44.725358
2,M00003,3,3,3,3,3,3,3,3,3,...,3,3,3,39,"[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]",-0.434366,0.275357,-0.418584,0.032791,50.327912
3,M00004,4,4,4,4,4,4,4,4,4,...,4,4,4,52,"[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]",0.817756,0.948237,0.236383,0.577263,55.772627
4,M00005,5,5,5,5,5,5,5,5,5,...,5,5,5,65,"[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5]",2.541674,2.173227,1.639552,1.291563,62.915626


In [12]:
out.columns

Index(['physician_id', 'F1_1_enjoymentoflife', 'F1_2_financialsufficiency',
       'F1_3_accesstoinformation', 'F1_4_leisureopportunities',
       'F1_5_mobilitypast2weeks', 'F1_6_accesstohealthservices',
       'F2_1_technicaltraining', 'F2_2_mentalhealthsupport',
       'F2_3_coworkersupportnetwork', 'F2_4_educationalhandlingoferrors',
       'F3_1_stresshurtsperformance', 'F3_2_stressledtoerrors',
       'F3_3_stresshurtsrelationships', 'total_score', 'responses_vector',
       'theta1_quality_of_life', 'theta2_institutional_support',
       'theta3_perceived_stress', 'theta_global', 'T_score_global'],
      dtype='object')

In [13]:
df_respostas_importacao = pd.read_excel('teste_lote_medicos.xlsx')
df_respostas_importacao.rename(
  columns={
    'Q28_sentidoultimasR3': 'F1_1_enjoymentoflife', 
    'Q29_capazultimasR3': 'F1_2_financialsufficiency',
    'Q29_capazultimasR4': 'F1_3_accesstoinformation', 
    'Q29_capazultimasR5': 'F1_4_leisureopportunities', 
    'Q30_locomoverultimas': 'F1_5_mobilitypast2weeks',
    'Q31_satisfeitovidaR9': 'F1_6_accesstohealthservices', 
    'Q59_percepcaodotrabalhoR1': 'F2_1_technicaltraining',
    'Q59_percepcaodotrabalhoR2': 'F2_2_mentalhealthsupport', 
    'Q59_percepcaodotrabalhoR3': 'F2_3_coworkersupportnetwork',
    'Q59_percepcaodotrabalhoR4': 'F2_4_educationalhandlingoferrors', 
    'Q60_estresseR1': 'F3_1_stresshurtsperformance', 
    'Q60_estresseR2': 'F3_2_stressledtoerrors',
    'Q60_estresseR3': 'F3_3_stresshurtsrelationships',
    'theta_global': 'theta_global_gabarito',
    'T_score_global': 'T_score_global_gabarito'
  }, inplace=True
)
df_respostas_importacao.columns

#Tirar os nulos
df_respostas_importacao = df_respostas_importacao.dropna()

In [14]:
columns_extract = ['F1_1_enjoymentoflife', 'F1_2_financialsufficiency',
       'F1_3_accesstoinformation', 'F1_4_leisureopportunities',
       'F1_5_mobilitypast2weeks', 'F1_6_accesstohealthservices',
       'F2_1_technicaltraining', 'F2_2_mentalhealthsupport',
       'F2_3_coworkersupportnetwork', 'F2_4_educationalhandlingoferrors',
       'F3_1_stresshurtsperformance', 'F3_2_stressledtoerrors',
       'F3_3_stresshurtsrelationships']

In [15]:
for col in columns_extract:
  df_respostas_importacao[col] = (
      df_respostas_importacao[col]
      .str.extract(r"\((\d+)\)")
      .astype(int)
  )
df_respostas_importacao

,physician_id,F1_1_enjoymentoflife,F1_2_financialsufficiency,F1_3_accesstoinformation,F1_4_leisureopportunities,F1_5_mobilitypast2weeks,F1_6_accesstohealthservices,F2_1_technicaltraining,F2_2_mentalhealthsupport,F2_3_coworkersupportnetwork,F2_4_educationalhandlingoferrors,F3_1_stresshurtsperformance,F3_2_stressledtoerrors,F3_3_stresshurtsrelationships,theta_F1,theta_F2,theta_F3,theta_global_gabarito,T_score_global_gabarito
0,124847127,4,5,4,3,5,4,1,1,1,1,1,2,1,0.722977,-1.520615,-1.515554,0.186538,51.865382
2,123603065,4,4,4,4,5,5,5,4,4,4,1,1,1,1.146841,1.143866,-1.882549,1.344137,63.441366
3,123738163,2,4,4,2,5,4,1,1,1,1,2,2,4,-0.441649,-1.520615,-0.483839,-0.552918,44.470816
4,124949534,1,2,4,1,2,1,1,1,1,1,5,2,5,-2.392374,-1.520615,0.929481,-1.706653,32.933470
7,123860541,3,5,4,3,4,5,4,4,4,4,1,1,1,0.351759,0.948237,-1.882549,0.963751,59.637505
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
93,124661611,2,2,3,3,4,3,1,3,1,1,4,1,5,-0.839552,-0.804633,0.262621,-0.672382,43.276178
96,124372298,2,3,2,2,5,4,1,4,1,1,5,4,1,-0.932455,-0.759385,0.322478,-0.710193,42.898069
97,124813099,2,3,4,2,4,1,1,1,1,2,4,4,5,-1.057470,-0.961866,0.653979,-0.916759,40.832411
98,124795949,2,3,2,2,5,2,1,1,1,1,4,1,4,-1.175540,-1.520615,-0.129104,-0.939271,40.607292


In [16]:
out_importacao = calc.score_batch(df_respostas_importacao)
out_importacao[[
  'theta_F1', 'theta1_quality_of_life',
  'theta_F2', 'theta2_institutional_support', 
  'theta_F3', 'theta3_perceived_stress',
  'theta_global_gabarito', 'theta_global',
  'T_score_global_gabarito', 'T_score_global'
]]

,theta_F1,theta1_quality_of_life,theta_F2,theta2_institutional_support,theta_F3,theta3_perceived_stress,theta_global_gabarito,theta_global,T_score_global_gabarito,T_score_global
0,0.722977,0.722977,-1.520615,-1.520615,-1.515554,-1.515554,0.186538,0.186538,51.865382,51.865382
2,1.146841,1.146841,1.143866,1.143866,-1.882549,-1.882549,1.344137,1.344137,63.441366,63.441366
3,-0.441649,-0.441649,-1.520615,-1.520615,-0.483839,-0.483839,-0.552918,-0.552918,44.470816,44.470816
4,-2.392374,-2.392374,-1.520615,-1.520615,0.929481,0.929481,-1.706653,-1.706653,32.933470,32.933470
7,0.351759,0.351759,0.948237,0.948237,-1.882549,-1.882549,0.963751,0.963751,59.637505,59.637505
...,...,...,...,...,...,...,...,...,...,...
93,-0.839552,-0.839552,-0.804633,-0.804633,0.262621,0.262621,-0.672382,-0.672382,43.276178,43.276178
96,-0.932455,-0.932455,-0.759385,-0.759385,0.322478,0.322478,-0.710193,-0.710193,42.898069,42.898069
97,-1.057470,-1.057470,-0.961866,-0.961866,0.653979,0.653979,-0.916759,-0.916759,40.832411,40.832411
98,-1.175540,-1.175540,-1.520615,-1.520615,-0.129104,-0.129104,-0.939271,-0.939271,40.607292,40.607292


---

# Afya MedQoL Student — exemplo de uso (estudantes de medicina)

Índice Afya MedQoL Student (8 itens, modelo bifatorial) para estudantes de medicina.
Cobre os mesmos três casos: respondente único, lote e salvamento em CSV.

In [17]:
from afya_medqol import MedQoLStudentCalculator

calc_estudante = MedQoLStudentCalculator()
print(calc_estudante.factor_items)

{'Factor1': ['F1_1_overallqol', 'F1_2_satisfactionwithhealth', 'F1_3_enjoymentoflife', 'F1_4_perceivedmeaninginlife'], 'Factor2': ['F2_1_energyfordailyactivities', 'F2_2_satisfactionwithsleep'], 'Factor3': ['F3_1_performdailyactivities', 'F3_2_capacityforwork']}


In [18]:
calc = MedQoLStudentCalculator()
perguntas_en = calc.item_questions(lang="en")  
perguntas_en 

{'F1_1_overallqol': '22. How would you rate your quality of life?',
 'F1_2_satisfactionwithhealth': '23. How satisfied are you with your health?',
 'F1_3_enjoymentoflife': '24.3 How much do you enjoy life?',
 'F1_4_perceivedmeaninginlife': '24.4 To what extent do you feel your life to be meaningful?',
 'F2_1_energyfordailyactivities': '25.1 Do you have enough energy for everyday life?',
 'F2_2_satisfactionwithsleep': '27.1 How satisfied are you with your sleep?',
 'F3_1_performdailyactivities': '27.2 How satisfied are you with your ability to perform your daily living activities?',
 'F3_2_capacityforwork': '27.3 How satisfied are you with your capacity for work?'}

## 1. Escoragem de um único estudante

In [19]:
calc_estudante = MedQoLStudentCalculator()

respostas_estudante = {
    "F1_1_overallqol": 1,
    "F1_2_satisfactionwithhealth": 1,
    "F1_3_enjoymentoflife": 1,
    "F1_4_perceivedmeaninginlife":1,
    "F2_1_energyfordailyactivities": 1,
    "F2_2_satisfactionwithsleep": 1,
    "F3_1_performdailyactivities": 1,
    "F3_2_capacityforwork":1
}

resultado_estudante = calc_estudante.score_student(respostas_estudante)

for chave in (
    "theta1_psychological_well_being", "theta2_vitality", "theta3_perceived_functional_capacity", "theta_global",
    "T_score_global",
):
    print(f"{chave:42s} {resultado_estudante[chave]}")

theta1_psychological_well_being            -1.0561838999736812
theta2_vitality                            0.050282711972277266
theta3_perceived_functional_capacity       -0.2547822423757723
theta_global                               -0.5984582713236506
T_score_global                             32.297937460166125


In [20]:
perguntas_estudante_en = calc_estudante.item_questions(lang='pt')
perguntas_estudante_en

{'F1_1_overallqol': '22. Pensando nas duas últimas semanas, como você avaliaria sua qualidade de vida?',
 'F1_2_satisfactionwithhealth': '23. Pensando nas duas últimas semanas, quão satisfeito(a) você está com a sua saúde?',
 'F1_3_enjoymentoflife': '24.3 O quanto você aproveita a vida?',
 'F1_4_perceivedmeaninginlife': '24.4 Em que medida você acha que a sua vida tem sentido?',
 'F2_1_energyfordailyactivities': '25.1 Você tem energia suficiente para seu dia-a-dia?',
 'F2_2_satisfactionwithsleep': '27.1 Quão satisfeito(a) você está com o seu sono?',
 'F3_1_performdailyactivities': '27.2 Quão satisfeito(a) você está com sua capacidade de desempenhar as atividades do seu dia-a-dia?',
 'F3_2_capacityforwork': '27.3 Quão satisfeito(a) você está com sua capacidade para o trabalho?'}

In [21]:
alternativas_estudante_pt = calc_estudante.item_options(lang="pt")
alternativas_estudante_pt

{'F1_1_overallqol': ['Muito ruim (1)',
  'Ruim (2)',
  'Nem ruim, nem boa (3)',
  'Boa (4)',
  'Muito boa (5)'],
 'F1_2_satisfactionwithhealth': ['Muito insatisfeito (1)',
  'Insatisfeito (2)',
  'Nem satisfeito, nem insatisfeito (3)',
  'Satisfeito (4)',
  'Muito satisfeito (5)'],
 'F1_3_enjoymentoflife': ['Nada (1)',
  'Muito pouco (2)',
  'Mais ou menos (3)',
  'Bastante (4)',
  'Extremamente (5)'],
 'F1_4_perceivedmeaninginlife': ['Nada (1)',
  'Muito pouco (2)',
  'Mais ou menos (3)',
  'Bastante (4)',
  'Extremamente (5)'],
 'F2_1_energyfordailyactivities': ['Nada (1)',
  'Muito pouco (2)',
  'Médio (3)',
  'Muito (4)',
  'Completamente (5)'],
 'F2_2_satisfactionwithsleep': ['Muito insatisfeito (1)',
  'Insatisfeito (2)',
  'Nem satisfeito, nem insatisfeito (3)',
  'Satisfeito (4)',
  'Muito satisfeito (5)'],
 'F3_1_performdailyactivities': ['Muito insatisfeito (1)',
  'Insatisfeito (2)',
  'Nem satisfeito, nem insatisfeito (3)',
  'Satisfeito (4)',
  'Muito satisfeito (5)'],
 'F

## 2. Escoragem em lote

In [22]:
df_estudantes = pd.DataFrame([
    {"student_id": "A001", "F1_1_overallqol": 5, "F1_2_satisfactionwithhealth": 5, "F1_3_enjoymentoflife": 5, "F1_4_perceivedmeaninginlife": 5,
     "F2_1_energyfordailyactivities": 5, "F2_2_satisfactionwithsleep": 5, "F3_1_performdailyactivities": 5, "F3_2_capacityforwork": 5},
    {"student_id": "A002", "F1_1_overallqol": 1, "F1_2_satisfactionwithhealth": 1, "F1_3_enjoymentoflife": 1, "F1_4_perceivedmeaninginlife": 1,
     "F2_1_energyfordailyactivities": 1, "F2_2_satisfactionwithsleep": 1, "F3_1_performdailyactivities": 1, "F3_2_capacityforwork": 1},
    {"student_id": "A003", "F1_1_overallqol": 3, "F1_2_satisfactionwithhealth": 3, "F1_3_enjoymentoflife": 3, "F1_4_perceivedmeaninginlife": 3,
     "F2_1_energyfordailyactivities": 3, "F2_2_satisfactionwithsleep": 3, "F3_1_performdailyactivities": 3, "F3_2_capacityforwork": 3},
])

df_resultado_estudantes = calc_estudante.score_batch(df_estudantes)
df_resultado_estudantes[["student_id", "theta1_psychological_well_being", "theta2_vitality",
                         "theta3_perceived_functional_capacity", "theta_global", "T_score_global"]]

,student_id,theta1_psychological_well_being,theta2_vitality,theta3_perceived_functional_capacity,theta_global,T_score_global
0,A001,0.546408,0.036023,0.311684,0.380536,61.294682
1,A002,-1.056184,0.050283,-0.254782,-0.598458,32.297937
2,A003,-0.617983,0.019531,-0.081657,-0.329279,40.270727


In [23]:
import numpy as np
import pandas as pd

colunas_estudantes = [
    "F1_1_overallqol",
    "F1_2_satisfactionwithhealth",
    "F1_3_enjoymentoflife",
    "F1_4_perceivedmeaninginlife",
    "F2_1_energyfordailyactivities",
    "F2_2_satisfactionwithsleep",
    "F3_1_performdailyactivities",
    "F3_2_capacityforwork",
]


def gerar_dataset_estudantes(n_amostras=5000, random_state=42):

    rng = np.random.default_rng(random_state)

    dados = {}

    # Distribuição equilibrada entre notas 1 a 5
    for coluna in colunas_estudantes:
        valores = np.tile(
            np.arange(1, 6),
            int(np.ceil(n_amostras / 5))
        )[:n_amostras]

        rng.shuffle(valores)
        dados[coluna] = valores

    df = pd.DataFrame(dados)

    # Perfis extremos e intermediários
    extremos = pd.DataFrame([
        [1]*8,  # baixa qualidade de vida
        [2]*8,
        [3]*8,  # intermediário
        [4]*8,
        [5]*8,  # alta qualidade de vida

        # perfis mistos
        [1,1,1,1,1,1,1,1],
        [5,5,5,5,5,5,1,1],
        [1,1,1,1,1,1,5,5],
        [5,1,5,1,5,1,5,1],
        [1,5,1,5,1,5,1,5],
    ], columns=colunas_estudantes)

    df = pd.concat(
        [extremos, df],
        ignore_index=True
    )

    # ID do estudante
    df.insert(
        0,
        "student_id",
        [f"A{i+1:05d}" for i in range(len(df))]
    )

    # Soma das respostas
    df["total_score"] = df[colunas_estudantes].sum(axis=1)

    # Vetor das respostas
    df["responses_vector"] = df[colunas_estudantes].apply(tuple, axis=1)

    return df


df_estudantes = gerar_dataset_estudantes(5000)

df_estudantes

,student_id,F1_1_overallqol,F1_2_satisfactionwithhealth,F1_3_enjoymentoflife,F1_4_perceivedmeaninginlife,F2_1_energyfordailyactivities,F2_2_satisfactionwithsleep,F3_1_performdailyactivities,F3_2_capacityforwork,total_score,responses_vector
0,A00001,1,1,1,1,1,1,1,1,8,"(1, 1, 1, 1, 1, 1, 1, 1)"
1,A00002,2,2,2,2,2,2,2,2,16,"(2, 2, 2, 2, 2, 2, 2, 2)"
2,A00003,3,3,3,3,3,3,3,3,24,"(3, 3, 3, 3, 3, 3, 3, 3)"
3,A00004,4,4,4,4,4,4,4,4,32,"(4, 4, 4, 4, 4, 4, 4, 4)"
4,A00005,5,5,5,5,5,5,5,5,40,"(5, 5, 5, 5, 5, 5, 5, 5)"
...,...,...,...,...,...,...,...,...,...,...,...
5005,A05006,3,1,5,1,5,1,2,2,20,"(3, 1, 5, 1, 5, 1, 2, 2)"
5006,A05007,2,5,1,4,1,5,4,3,25,"(2, 5, 1, 4, 1, 5, 4, 3)"
5007,A05008,5,5,2,5,1,5,5,2,30,"(5, 5, 2, 5, 1, 5, 5, 2)"
5008,A05009,3,2,4,1,3,5,4,5,27,"(3, 2, 4, 1, 3, 5, 4, 5)"


In [24]:
df_resultado_estudantes = calc_estudante.score_batch(df_estudantes)

df_resultado_estudantes[
    [
        "student_id",
        "theta1_psychological_well_being",
        "theta2_vitality",
        "theta3_perceived_functional_capacity",
        "theta_global",
        "T_score_global"
    ]
]

,student_id,theta1_psychological_well_being,theta2_vitality,theta3_perceived_functional_capacity,theta_global,T_score_global
0,A00001,-1.056184,0.050283,-0.254782,-0.598458,32.297937
1,A00002,-0.869018,0.162381,-0.058415,-0.420934,37.556006
2,A00003,-0.617983,0.019531,-0.081657,-0.329279,40.270727
3,A00004,-0.216177,-0.077333,0.056208,-0.101263,47.024325
4,A00005,0.546408,0.036023,0.311684,0.380536,61.294682
...,...,...,...,...,...,...
5005,A05006,-0.487551,-1.263203,-0.523564,-0.633515,31.259592
5006,A05007,-1.043382,1.495818,0.430291,-0.114003,46.646985
5007,A05008,0.834147,1.315404,0.170443,0.695417,70.621109
5008,A05009,-1.443806,0.887722,0.907009,-0.256704,42.420331


In [25]:
import plotly.express as px

fig = px.scatter(
    df_resultado_estudantes,
    x="total_score",
    y="T_score_global",
    hover_name="responses_vector",  # aparece em destaque ao passar o mouse
    hover_data={
        "total_score": True,
        "T_score_global": True,
        "responses_vector": False,   # evita duplicação, pois já está em hover_name
    },
    title="Enjoyment of Life vs Total Score"
)

fig.update_layout(
    xaxis_title="Total Score",
    yaxis_title="T_score_global",
    template="plotly_white"
)

fig.show()

In [26]:
fig = px.scatter(
    df_resultado_estudantes,
    x="total_score",
    y="theta1_psychological_well_being",
    hover_name="responses_vector",  # aparece em destaque ao passar o mouse
    hover_data={
        "total_score": True,
        "theta1_psychological_well_being": True,
        "responses_vector": False,   # evita duplicação, pois já está em hover_name
    },
    title="theta1_psychological_well_being vs Total Score"
)

fig.update_layout(
    xaxis_title="Total Score",
    yaxis_title="theta1_psychological_well_being",
    template="plotly_white"
)

fig.show()

In [27]:
fig = px.scatter(
    df_resultado_estudantes,
    x="total_score",
    y="theta2_vitality",
    hover_name="responses_vector",  # aparece em destaque ao passar o mouse
    hover_data={
        "total_score": True,
        "theta2_vitality": True,
        "responses_vector": False,   # evita duplicação, pois já está em hover_name
    },
    title="theta2_vitality vs Total Score"
)

fig.update_layout(
    xaxis_title="Total Score",
    yaxis_title="theta2_vitality",
    template="plotly_white"
)

fig.show()

In [28]:
fig = px.scatter(
    df_resultado_estudantes,
    x="total_score",
    y="theta3_perceived_functional_capacity",
    hover_name="responses_vector",  # aparece em destaque ao passar o mouse
    hover_data={
        "total_score": True,
        "theta3_perceived_functional_capacity": True,
        "responses_vector": False,   # evita duplicação, pois já está em hover_name
    },
    title="theta3_perceived_functional_capacity vs Total Score"
)

fig.update_layout(
    xaxis_title="Total Score",
    yaxis_title="theta3_perceived_functional_capacity",
    template="plotly_white"
)

fig.show()

In [29]:
df_estudantes_importacao = pd.read_excel('teste_lote_estudantes.xlsx')
df_estudantes_importacao.rename(
  columns={"Q22_percepcaoqualidadedevida":"F1_1_overallqol", 
   "Q23_satisfacaosaude":"F1_2_satisfactionwithhealth",
    "Q24_sentidoultimas2sR3":"F1_3_enjoymentoflife", 
    "Q24_sentidoultimas2sR4":"F1_4_perceivedmeaninginlife",
    "Q25_capazultimas2sR1":"F2_1_energyfordailyactivities", 
    "Q27_satisfeitovida2sR1":"F2_2_satisfactionwithsleep",
    "Q27_satisfeitovida2sR2":"F3_1_performdailyactivities", 
    "Q27_satisfeitovida2sR3":"F3_2_capacityforwork",
    "theta_global":"theta_global_gabarito",
    "T_score_global": "T_score_global_gabarito"}
, inplace=True)

df_estudantes_importacao.columns

Index(['student_id', 'F1_1_overallqol', 'F1_2_satisfactionwithhealth',
       'F1_3_enjoymentoflife', 'F1_4_perceivedmeaninginlife',
       'F2_1_energyfordailyactivities', 'F2_2_satisfactionwithsleep',
       'F3_1_performdailyactivities', 'F3_2_capacityforwork',
       'theta_bem_estar_psicologico', 'theta_vitalidade',
       'theta_capacidade_funcional', 'theta_global_gabarito',
       'T_score_global_gabarito'],
      dtype='object')

In [30]:
df_resultado_estudantes = calc_estudante.score_batch(df_estudantes_importacao)
df_resultado_estudantes[["student_id", "F1_1_overallqol",
    "F1_2_satisfactionwithhealth",
    "F1_3_enjoymentoflife",
    "F1_4_perceivedmeaninginlife",
    "F2_1_energyfordailyactivities",
    "F2_2_satisfactionwithsleep",
    "F3_1_performdailyactivities",
    "F3_2_capacityforwork",
    "theta_bem_estar_psicologico", "theta1_psychological_well_being", "theta_vitalidade", "theta2_vitality",
    "theta_capacidade_funcional", "theta3_perceived_functional_capacity", "theta_global_gabarito", "theta_global", "T_score_global_gabarito", "T_score_global"]]

,student_id,F1_1_overallqol,F1_2_satisfactionwithhealth,F1_3_enjoymentoflife,F1_4_perceivedmeaninginlife,F2_1_energyfordailyactivities,F2_2_satisfactionwithsleep,F3_1_performdailyactivities,F3_2_capacityforwork,theta_bem_estar_psicologico,theta1_psychological_well_being,theta_vitalidade,theta2_vitality,theta_capacidade_funcional,theta3_perceived_functional_capacity,theta_global_gabarito,theta_global,T_score_global_gabarito,T_score_global
0,125066552,5,4,4,5,3,2,4,4,1.178212,1.178212,-0.413414,-0.413414,0.518855,0.518855,0.684746,0.684746,70.305,70.305060
1,125066557,5,3,4,5,3,3,4,4,0.791919,0.791919,-0.092090,-0.092090,0.469886,0.469886,0.532780,0.532780,65.804,65.803989
2,125066604,5,4,4,5,4,4,3,4,0.829259,0.829259,-0.078368,-0.078368,-0.726005,-0.726005,0.152963,0.152963,54.554,54.554224
3,125066628,5,5,4,5,5,5,5,5,0.184556,0.184556,0.042120,0.042120,0.431823,0.431823,0.243076,0.243076,57.223,57.223270
4,125066630,5,5,5,5,5,5,5,5,0.546408,0.546408,0.036023,0.036023,0.311684,0.311684,0.380536,0.380536,61.295,61.294682
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,125459927,2,3,2,3,3,2,3,4,-1.182289,-1.182289,-0.417462,-0.417462,0.639294,0.639294,-0.441691,-0.441691,36.941,36.941227
96,125459944,1,2,4,4,2,3,4,4,-1.246927,-1.246927,0.483080,0.483080,1.109535,1.109535,-0.161198,-0.161198,45.249,45.249113
97,125459956,4,3,4,4,3,3,4,4,-0.016188,-0.016188,-0.071387,-0.071387,0.633477,0.633477,0.191939,0.191939,55.709,55.708656
98,125459960,4,2,4,3,3,3,3,3,0.007056,0.007056,0.006710,0.006710,-0.166910,-0.166910,-0.051275,-0.051275,48.505,48.504900


## 3. Lendo direto de um CSV com score_batch

Mesma ideia: `score_batch` só aceita um `DataFrame` já carregado — para
pontuar direto de um CSV, leia com `pandas.read_csv(...)` primeiro. Sem
salvar nada — use `out.to_csv(...)` ou a linha de comando
(`iqol-estudante respostas.csv --saida resultado.csv`) para persistir.

In [31]:
out_estudante = calc_estudante.score_batch(df_estudantes)
out_estudante.head()

,student_id,F1_1_overallqol,F1_2_satisfactionwithhealth,F1_3_enjoymentoflife,F1_4_perceivedmeaninginlife,F2_1_energyfordailyactivities,F2_2_satisfactionwithsleep,F3_1_performdailyactivities,F3_2_capacityforwork,total_score,responses_vector,theta1_psychological_well_being,theta2_vitality,theta3_perceived_functional_capacity,theta_global,T_score_global
0,A00001,1,1,1,1,1,1,1,1,8,"(1, 1, 1, 1, 1, 1, 1, 1)",-1.056184,0.050283,-0.254782,-0.598458,32.297937
1,A00002,2,2,2,2,2,2,2,2,16,"(2, 2, 2, 2, 2, 2, 2, 2)",-0.869018,0.162381,-0.058415,-0.420934,37.556006
2,A00003,3,3,3,3,3,3,3,3,24,"(3, 3, 3, 3, 3, 3, 3, 3)",-0.617983,0.019531,-0.081657,-0.329279,40.270727
3,A00004,4,4,4,4,4,4,4,4,32,"(4, 4, 4, 4, 4, 4, 4, 4)",-0.216177,-0.077333,0.056208,-0.101263,47.024325
4,A00005,5,5,5,5,5,5,5,5,40,"(5, 5, 5, 5, 5, 5, 5, 5)",0.546408,0.036023,0.311684,0.380536,61.294682
